In [1]:
# ============================================================
# AURORA-TWETF Notebook 21 + Notebook 22, revised
# Single-cell Colab version
#
# Notebook 21:
#   S43  Formal bootstrap equivalence tests
#   S43b Equivalence summary
#   S44  Pre-evaluation constant-lambda calibration
#   S44b Pre-evaluation constant-lambda performance
#   S45  Optimizer reliability diagnostics
#   S45b Constraint-binding diagnostics
#
# Notebook 22:
#   S46  Cash-return / nonzero-risk-free-rate sensitivity
#   S46b Compact cash-return sensitivity summary
#
# Revisions:
# 1. Fixes pre-evaluation lambda failure by loading raw ETF files only from:
#    /content/drive/MyDrive/AURORA_TWETF/data/raw_yfinance/
# 2. Prevents invalid S46 source-aware rows by default.
#    Source-aware cash-return sensitivity is skipped unless exact source-aware
#    cash-weight files are deliberately supplied.
# 3. Keeps only valid n=319 Notebook18 dynamic / constant-lambda diagnostic rows
#    in S46 by default.
#
# Research diagnostics only. Not financial advice.
# ============================================================

from __future__ import annotations

import json
import math
import re
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or already mounted.")

import numpy as np
import pandas as pd

try:
    from scipy.optimize import minimize
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False
    print("scipy.optimize unavailable. Pre-evaluation lambda reconstruction will be skipped.")

# ============================================================
# 0. User settings
# ============================================================

PROJECT_CODE = "AURORA_TWETF"
PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")
OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"

DATA_ROOT = PUBLICATION_ROOT / "data"
RAW_YF_DIR = DATA_ROOT / "raw_yfinance"
MODELING_DIR = DATA_ROOT / "modeling"

STRICT_START = pd.Timestamp("2024-11-27")
STRICT_END = pd.Timestamp("2026-03-25")

ANNUALIZATION_DAYS = 252

BOOTSTRAP_REPLICATIONS = 5000
BOOTSTRAP_BLOCK_LENGTH = 20
BOOTSTRAP_RANDOM_SEED = 20260722

EQUIV_MARGINS = {
    "total_return": 0.01,
    "sharpe": 0.05,
    "sortino": 0.10,
    "max_drawdown": 0.01,
    "avg_cash": 0.01,
    "cash_cap_binding": 0.01,
}

LAMBDA_GRID = np.round(np.arange(0.0, 100.0 + 1e-9, 1.0), 4)

CASH_CAP = 0.60
GENERAL_ETF_CAP = 0.45
ETF_00881_CAP = 0.30

TRANSACTION_COST_BPS = 10
TRANSACTION_COST_RATE = TRANSACTION_COST_BPS / 10000.0

ALPHA_20 = 0.30
ALPHA_60 = 0.70
LAMBDA0 = 10.0
GAMMA_U = 2.5
GAMMA_B = 2.0
RHO = 0.25

MU_LOOKBACK = 63
COV_LOOKBACK = 126
MEAN_SHRINKAGE = 0.60
MOMENTUM_WEIGHT = 0.40

REBALANCE_CONVENTION = "month_end"

# Important: keep False unless you have exact source-aware cash-weight files.
INCLUDE_SOURCE_AWARE_CASH_SENSITIVITY = False

CASH_RATE_SCENARIOS = [
    {"scenario": "zero_cash_return", "annual_cash_return": 0.0000, "annual_risk_free_rate": 0.0000},
    {"scenario": "low_cash_yield_1pct", "annual_cash_return": 0.0100, "annual_risk_free_rate": 0.0100},
    {"scenario": "mid_cash_yield_1p5pct", "annual_cash_return": 0.0150, "annual_risk_free_rate": 0.0150},
    {"scenario": "higher_cash_yield_2pct", "annual_cash_return": 0.0200, "annual_risk_free_rate": 0.0200},
]

# Manual overrides if needed.
MANUAL_NOTEBOOK18_RETURNS_PATH = None
MANUAL_NOTEBOOK18_WEIGHTS_PATH = None
MANUAL_NOTEBOOK18_DIAGNOSTICS_PATH = None
MANUAL_SOURCE_AWARE_RETURN_MATRIX_PATH = None
MANUAL_NOTEBOOK08_INPUT_INDEX = None

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = OUTPUT_ROOT / "equivalence_lambda_cash_sensitivity" / f"run_{RUN_ID}"
TABLE_RUN_DIR = RUN_ROOT / "tables"
REPORT_RUN_DIR = RUN_ROOT / "reports"
DIAG_DIR = RUN_ROOT / "diagnostics"
RETURNS_DIR = RUN_ROOT / "returns"
WEIGHTS_DIR = RUN_ROOT / "weights"

for d in [RUN_ROOT, TABLE_RUN_DIR, REPORT_RUN_DIR, DIAG_DIR, RETURNS_DIR, WEIGHTS_DIR, TABLE_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("AURORA-TWETF Revised Notebooks 21 + 22")
print("RUN_ID:", RUN_ID)
print("RUN_ROOT:", RUN_ROOT)
print("=" * 100)

# ============================================================
# 1. General utilities
# ============================================================

def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    path = Path(path)
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root):
    rows = []
    root = Path(root)
    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(stat.st_mtime, timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })
    return pd.DataFrame(rows)

def write_table(df, filename_stem, index=False):
    local_csv = TABLE_RUN_DIR / f"{filename_stem}.csv"
    global_csv = TABLE_DIR / f"{filename_stem}_{RUN_ID}.csv"
    df.to_csv(local_csv, index=index)
    df.to_csv(global_csv, index=index)
    print("Saved:", local_csv)
    print("Saved:", global_csv)
    return local_csv, global_csv

def write_rounded_table(df, filename_stem, digits=6):
    out = df.copy()
    for c in out.select_dtypes(include=[np.number]).columns:
        out[c] = out[c].round(digits)
    return write_table(out, f"{filename_stem}_rounded", index=False)

def normalize_name(x):
    return re.sub(r"[^A-Za-z0-9]+", "", str(x)).lower()

def looks_like_date_series(s):
    parsed = pd.to_datetime(s, errors="coerce")
    if not isinstance(parsed, pd.Series):
        parsed = pd.Series(parsed)
    if parsed.notna().mean() < 0.50:
        return False, parsed
    years = parsed.dt.year
    if years.between(1990, 2035).mean() < 0.50:
        return False, parsed
    if parsed.nunique(dropna=True) < min(10, max(2, len(parsed) // 10)):
        return False, parsed
    return True, parsed

def set_datetime_index_flex(df):
    df = df.copy()

    if isinstance(df.index, pd.DatetimeIndex):
        years = pd.Series(df.index.year)
        if years.between(1990, 2035).mean() > 0.50:
            df.index = pd.to_datetime(df.index)
            df.index.name = "date"
            return df.sort_index()

    preferred = ["date", "Date", "DATE", "datetime", "Datetime", "timestamp", "Timestamp", "Unnamed: 0", "index", "Index"]
    candidate_cols = [c for c in preferred if c in df.columns] + [c for c in df.columns if c not in preferred]

    for c in candidate_cols:
        try:
            ok, parsed = looks_like_date_series(df[c])
            if ok:
                df = df.drop(columns=[c])
                df.index = pd.to_datetime(parsed)
                df.index.name = "date"
                return df[~df.index.isna()].sort_index()
        except Exception:
            pass

    idx_series = pd.Series(df.index)
    ok, parsed_idx = looks_like_date_series(idx_series)
    if ok:
        df.index = pd.to_datetime(parsed_idx.values)
        df.index.name = "date"
        return df[~df.index.isna()].sort_index()

    return df

def read_table_auto(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path, low_memory=False)
    else:
        raise ValueError(f"Unsupported file type: {path}")
    return set_datetime_index_flex(df)

def save_frame(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() == ".parquet":
        try:
            df.to_parquet(path)
        except Exception:
            df.to_csv(path.with_suffix(".csv"))
    else:
        df.to_csv(path)

def find_files(patterns, roots, max_files=None):
    if isinstance(patterns, str):
        patterns = [patterns]
    if isinstance(roots, (str, Path)):
        roots = [roots]

    out = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for pat in patterns:
            out.extend(list(root.rglob(pat)))

    out = sorted(list(set([p for p in out if p.exists() and p.is_file()])), key=lambda p: p.stat().st_mtime, reverse=True)
    if max_files:
        return out[:max_files]
    return out

def first_file(patterns, roots):
    files = find_files(patterns, roots, max_files=1)
    return files[0] if files else None

# ============================================================
# 2. Performance utilities
# ============================================================

def performance_metrics_from_returns(r, rf_daily=0.0):
    r = pd.Series(r).dropna().astype(float)
    n = len(r)
    if n == 0:
        return {k: np.nan for k in ["n_days", "total_return", "annual_return", "annual_volatility", "sharpe", "sortino", "max_drawdown", "calmar"]}

    excess = r - rf_daily
    equity = (1.0 + r).cumprod()
    dd = equity / equity.cummax() - 1.0

    total_return = float(equity.iloc[-1] - 1.0)
    annual_return = float(equity.iloc[-1] ** (ANNUALIZATION_DAYS / max(n, 1)) - 1.0)
    daily_vol = float(excess.std(ddof=1)) if n > 1 else np.nan
    annual_vol = float(daily_vol * np.sqrt(ANNUALIZATION_DAYS)) if np.isfinite(daily_vol) else np.nan
    sharpe = float(excess.mean() / daily_vol * np.sqrt(ANNUALIZATION_DAYS)) if np.isfinite(daily_vol) and daily_vol > 0 else np.nan

    downside = excess[excess < 0]
    downside_vol = float(downside.std(ddof=1)) if len(downside) > 1 else np.nan
    sortino = float(excess.mean() / downside_vol * np.sqrt(ANNUALIZATION_DAYS)) if np.isfinite(downside_vol) and downside_vol > 0 else np.nan

    max_dd = float(dd.min())
    calmar = float(annual_return / abs(max_dd)) if max_dd < 0 else np.nan

    return {
        "n_days": int(n),
        "start_date": str(r.index.min().date()) if isinstance(r.index, pd.DatetimeIndex) else "",
        "end_date": str(r.index.max().date()) if isinstance(r.index, pd.DatetimeIndex) else "",
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown": max_dd,
        "calmar": calmar,
        "avg_daily_return": float(r.mean()),
        "daily_volatility": float(r.std(ddof=1)) if n > 1 else np.nan,
        "hit_rate": float((r > 0).mean()),
    }

def paired_metric_diffs(a, b):
    a = pd.Series(a).dropna().astype(float)
    b = pd.Series(b).dropna().astype(float)
    common = a.index.intersection(b.index).sort_values()
    if len(common) > 0:
        a = a.loc[common]
        b = b.loc[common]
    else:
        if len(a) != len(b):
            raise ValueError("No common index and unequal lengths.")
        a = a.reset_index(drop=True)
        b = b.reset_index(drop=True)

    ma = performance_metrics_from_returns(a)
    mb = performance_metrics_from_returns(b)

    return {
        "total_return": ma["total_return"] - mb["total_return"],
        "sharpe": ma["sharpe"] - mb["sharpe"],
        "sortino": ma["sortino"] - mb["sortino"],
        "max_drawdown": ma["max_drawdown"] - mb["max_drawdown"],
    }

def circular_block_indices(n, block_length, rng):
    idx = []
    while len(idx) < n:
        start = int(rng.integers(0, n))
        idx.extend([(start + j) % n for j in range(block_length)])
    return np.asarray(idx[:n], dtype=int)

def bootstrap_metric_diffs(a, b, n_rep=5000, block_length=20, seed=42):
    a = pd.Series(a).dropna().astype(float)
    b = pd.Series(b).dropna().astype(float)
    common = a.index.intersection(b.index).sort_values()

    if len(common) > 0:
        a = a.loc[common]
        b = b.loc[common]
    else:
        if len(a) != len(b):
            raise ValueError("No common index and unequal lengths.")
        a = a.reset_index(drop=True)
        b = b.reset_index(drop=True)

    n = len(a)
    rng = np.random.default_rng(seed)
    obs = paired_metric_diffs(a, b)

    dist = {m: [] for m in obs.keys()}
    av, bv = a.values, b.values

    for _ in range(n_rep):
        idx = circular_block_indices(n, block_length, rng)
        d = paired_metric_diffs(pd.Series(av[idx]), pd.Series(bv[idx]))
        for m, v in d.items():
            dist[m].append(v)

    rows = []
    for metric, vals in dist.items():
        arr = np.asarray(vals, dtype=float)
        arr = arr[np.isfinite(arr)]
        lo, hi = np.percentile(arr, [2.5, 97.5])
        rows.append({
            "metric": metric,
            "observed_difference": float(obs[metric]),
            "ci95_lower": float(lo),
            "ci95_upper": float(hi),
            "bootstrap_replications": n_rep,
            "block_length": block_length,
            "n_days": n,
        })
    return pd.DataFrame(rows)

def bootstrap_mean_diff(a, b, metric_name, n_rep=5000, block_length=20, seed=42):
    a = pd.Series(a).dropna().astype(float)
    b = pd.Series(b).dropna().astype(float)
    common = a.index.intersection(b.index).sort_values()

    if len(common) > 0:
        a = a.loc[common]
        b = b.loc[common]
    else:
        if len(a) != len(b):
            raise ValueError("No common index and unequal lengths.")
        a = a.reset_index(drop=True)
        b = b.reset_index(drop=True)

    diff = a - b
    n = len(diff)
    rng = np.random.default_rng(seed)
    dv = diff.values
    dist = []

    for _ in range(n_rep):
        idx = circular_block_indices(n, block_length, rng)
        dist.append(float(np.mean(dv[idx])))

    arr = np.asarray(dist)
    lo, hi = np.percentile(arr, [2.5, 97.5])

    return {
        "metric": metric_name,
        "observed_difference": float(diff.mean()),
        "ci95_lower": float(lo),
        "ci95_upper": float(hi),
        "bootstrap_replications": n_rep,
        "block_length": block_length,
        "n_days": n,
    }

def equivalence_status(row, margin):
    return "Equivalent within margin" if row["ci95_lower"] >= -margin and row["ci95_upper"] <= margin else "Not equivalent within margin"

# ============================================================
# 3. Strategy extraction utilities
# ============================================================

DYN_LABELS = ["Original dynamic AURORA", "Dynamic AURORA", "Original_dynamic_AURORA"]
CONST_LABELS = ["Exposure-matched constant-lambda AURORA", "Exposure matched constant lambda AURORA", "constant-lambda AURORA", "constant lambda AURORA"]
NOPROB_LABELS = [
    "Exposure-matched constant-lambda no-probability AURORA",
    "Exposure-matched no-probability constant-lambda AURORA",
    "No-probability constant-lambda AURORA",
    "constant-lambda no-probability AURORA",
    "no-probability constant-lambda",
]

def extract_strategy_series(df, label_candidates, value_col_candidates=None):
    if isinstance(label_candidates, str):
        label_candidates = [label_candidates]

    if value_col_candidates is None:
        value_col_candidates = ["net_return", "daily_return", "return", "returns", "strategy_return"]

    name_cols = ["strategy_control", "strategy_name", "strategy", "policy_name", "control", "label", "portfolio"]

    for name_col in name_cols:
        if name_col not in df.columns:
            continue
        ret_cols = [c for c in value_col_candidates if c in df.columns]
        if not ret_cols:
            continue
        ret_col = ret_cols[0]
        norm_names = df[name_col].astype(str).map(normalize_name)

        for lab in label_candidates:
            nlab = normalize_name(lab)
            mask = norm_names == nlab
            if mask.any():
                s = pd.to_numeric(df.loc[mask, ret_col], errors="coerce").dropna()
                s.index = pd.to_datetime(df.loc[mask].index)
                return s.sort_index()

        for lab in label_candidates:
            nlab = normalize_name(lab)
            mask = norm_names.map(lambda x: nlab in x or x in nlab)
            if mask.any():
                s = pd.to_numeric(df.loc[mask, ret_col], errors="coerce").dropna()
                s.index = pd.to_datetime(df.loc[mask].index)
                return s.sort_index()

    for lab in label_candidates:
        nlab = normalize_name(lab)
        for c in df.columns:
            nc = normalize_name(c)
            if nc == nlab or nlab in nc or nc in nlab:
                s = pd.to_numeric(df[c], errors="coerce").dropna()
                if not isinstance(s.index, pd.DatetimeIndex):
                    s.index = pd.to_datetime(s.index)
                return s.sort_index()

    raise ValueError(f"Could not extract strategy series for {label_candidates}")

def standardize_weight_columns(wdf):
    wdf = wdf.copy()
    rename = {}
    for c in wdf.columns:
        nc = normalize_name(c)
        if "006208" in nc:
            rename[c] = "006208"
        elif "00692" in nc:
            rename[c] = "00692"
        elif "00881" in nc:
            rename[c] = "00881"
        elif "0050" in nc:
            rename[c] = "0050"
        elif "cash" in nc:
            rename[c] = "cash"
    wdf = wdf.rename(columns=rename)
    keep = [c for c in ["0050", "006208", "00692", "00881", "cash"] if c in wdf.columns]
    return wdf[keep].apply(pd.to_numeric, errors="coerce").sort_index()

def extract_strategy_weights(df, label_candidates):
    if isinstance(label_candidates, str):
        label_candidates = [label_candidates]

    name_cols = ["strategy_control", "strategy_name", "strategy", "policy_name", "control", "label", "portfolio"]
    asset_cols = ["asset", "ticker", "symbol", "asset_name"]
    weight_cols = ["weight", "target_weight", "w", "portfolio_weight"]

    for name_col in name_cols:
        if name_col not in df.columns:
            continue

        asset_col = next((c for c in asset_cols if c in df.columns), None)
        weight_col = next((c for c in weight_cols if c in df.columns), None)

        if asset_col and weight_col:
            norm_names = df[name_col].astype(str).map(normalize_name)
            for lab in label_candidates:
                nlab = normalize_name(lab)
                mask = norm_names.map(lambda x: x == nlab or nlab in x or x in nlab)
                if mask.any():
                    sub = df.loc[mask].copy()
                    sub.index = pd.to_datetime(sub.index)
                    wide = sub.pivot_table(index=sub.index, columns=asset_col, values=weight_col, aggfunc="last")
                    return standardize_weight_columns(wide)

        possible_weight_cols = []
        for c in df.columns:
            nc = normalize_name(c)
            if any(tok in nc for tok in ["0050", "006208", "00692", "00881", "cash"]):
                possible_weight_cols.append(c)

        if possible_weight_cols:
            norm_names = df[name_col].astype(str).map(normalize_name)
            for lab in label_candidates:
                nlab = normalize_name(lab)
                mask = norm_names.map(lambda x: x == nlab or nlab in x or x in nlab)
                if mask.any():
                    sub = df.loc[mask, possible_weight_cols].copy()
                    sub.index = pd.to_datetime(sub.index)
                    return standardize_weight_columns(sub)

    selected = {}
    for lab in label_candidates:
        nlab = normalize_name(lab)
        for c in df.columns:
            nc = normalize_name(c)
            if nlab in nc:
                if "006208" in nc:
                    selected[c] = "006208"
                elif "00692" in nc:
                    selected[c] = "00692"
                elif "00881" in nc:
                    selected[c] = "00881"
                elif "0050" in nc:
                    selected[c] = "0050"
                elif "cash" in nc:
                    selected[c] = "cash"
    if selected:
        out = df[list(selected.keys())].copy()
        out.columns = [selected[c] for c in out.columns]
        out = out.groupby(level=0, axis=1).last()
        if not isinstance(out.index, pd.DatetimeIndex):
            out.index = pd.to_datetime(out.index)
        return standardize_weight_columns(out)

    possible = {}
    for c in df.columns:
        nc = normalize_name(c)
        if "006208" in nc:
            possible[c] = "006208"
        elif "00692" in nc:
            possible[c] = "00692"
        elif "00881" in nc:
            possible[c] = "00881"
        elif "0050" in nc:
            possible[c] = "0050"
        elif "cash" in nc:
            possible[c] = "cash"

    if len(possible) >= 2:
        out = df[list(possible.keys())].copy()
        out.columns = [possible[c] for c in out.columns]
        out = out.groupby(level=0, axis=1).last()
        if not isinstance(out.index, pd.DatetimeIndex):
            out.index = pd.to_datetime(out.index)
        return standardize_weight_columns(out)

    raise ValueError(f"Could not extract weights for {label_candidates}")

def cash_series_from_weights(wdf):
    wdf = standardize_weight_columns(wdf)
    if "cash" in wdf.columns:
        return pd.to_numeric(wdf["cash"], errors="coerce").dropna()
    risky_cols = [c for c in ["0050", "006208", "00692", "00881"] if c in wdf.columns]
    if risky_cols:
        return (1.0 - wdf[risky_cols].sum(axis=1)).dropna()
    raise ValueError("No cash or risky-weight columns found.")

# ============================================================
# 4. Locate files
# ============================================================

print("\n" + "=" * 100)
print("Locating prior output files")
print("=" * 100)

search_roots = [PUBLICATION_ROOT, OUTPUT_ROOT, OUTPUT_ROOT.parent]

n18_returns_path = Path(MANUAL_NOTEBOOK18_RETURNS_PATH) if MANUAL_NOTEBOOK18_RETURNS_PATH else first_file(["all_notebook18_returns.parquet", "all_notebook18_returns.csv"], search_roots)
n18_weights_path = Path(MANUAL_NOTEBOOK18_WEIGHTS_PATH) if MANUAL_NOTEBOOK18_WEIGHTS_PATH else first_file(["all_notebook18_weights.parquet", "all_notebook18_weights.csv"], search_roots)
n18_diag_path = Path(MANUAL_NOTEBOOK18_DIAGNOSTICS_PATH) if MANUAL_NOTEBOOK18_DIAGNOSTICS_PATH else first_file(["all_notebook18_diagnostics.parquet", "all_notebook18_diagnostics.csv"], search_roots)
source_matrix_path = Path(MANUAL_SOURCE_AWARE_RETURN_MATRIX_PATH) if MANUAL_SOURCE_AWARE_RETURN_MATRIX_PATH else first_file(
    ["notebook13B_source_aware_strict_test_return_matrix.parquet", "notebook13B_source_aware_strict_test_return_matrix.csv"],
    search_roots,
)
notebook08_input_index = Path(MANUAL_NOTEBOOK08_INPUT_INDEX) if MANUAL_NOTEBOOK08_INPUT_INDEX else first_file(["NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv"], search_roots)

print("Notebook18 returns:", n18_returns_path)
print("Notebook18 weights:", n18_weights_path)
print("Notebook18 diagnostics:", n18_diag_path)
print("Source-aware return matrix:", source_matrix_path)
print("Notebook08 input index:", notebook08_input_index)

warnings_list = []

# ============================================================
# 5. Notebook 21A: equivalence tests
# ============================================================

print("\n" + "=" * 100)
print("Notebook 21A: Formal bootstrap equivalence tests")
print("=" * 100)

equiv_rows = []

if n18_returns_path is None:
    warnings_list.append("Notebook18 returns file not found; S43 cannot be generated.")
    s43 = pd.DataFrame()
    s43b = pd.DataFrame()
else:
    n18_returns_df = read_table_auto(n18_returns_path)
    dyn_ret = extract_strategy_series(n18_returns_df, DYN_LABELS)
    const_ret = extract_strategy_series(n18_returns_df, CONST_LABELS)

    specs = [("Dynamic vs ex post constant-lambda", dyn_ret, const_ret, CONST_LABELS)]

    try:
        noprob_ret = extract_strategy_series(n18_returns_df, NOPROB_LABELS)
        specs.append(("Dynamic vs ex post no-probability constant-lambda", dyn_ret, noprob_ret, NOPROB_LABELS))
    except Exception as e:
        warnings_list.append(f"No-probability return series not found: {e}")

    weight_cache = {}

    if n18_weights_path is not None:
        try:
            n18_weights_df = read_table_auto(n18_weights_path)
            weight_cache["dynamic"] = extract_strategy_weights(n18_weights_df, DYN_LABELS)
            weight_cache["constant"] = extract_strategy_weights(n18_weights_df, CONST_LABELS)
            try:
                weight_cache["noprob"] = extract_strategy_weights(n18_weights_df, NOPROB_LABELS)
            except Exception as e:
                warnings_list.append(f"No-probability weights not found: {e}")
        except Exception as e:
            warnings_list.append(f"Could not parse Notebook18 weights: {e}")
    else:
        warnings_list.append("Notebook18 weights file not found; cash equivalence skipped.")

    for spec_idx, (comp_name, a, b, labels) in enumerate(specs):
        a = a.loc[(a.index >= STRICT_START) & (a.index <= STRICT_END)].dropna()
        b = b.loc[(b.index >= STRICT_START) & (b.index <= STRICT_END)].dropna()
        common = a.index.intersection(b.index).sort_values()
        a = a.loc[common]
        b = b.loc[common]
        print(comp_name, "n =", len(common))

        boot = bootstrap_metric_diffs(
            a, b,
            n_rep=BOOTSTRAP_REPLICATIONS,
            block_length=BOOTSTRAP_BLOCK_LENGTH,
            seed=BOOTSTRAP_RANDOM_SEED + spec_idx
        )

        for _, row in boot.iterrows():
            metric = row["metric"]
            margin = EQUIV_MARGINS[metric]
            out = row.to_dict()
            out["comparison"] = comp_name
            out["equivalence_margin"] = margin
            out["equivalence_result"] = equivalence_status(out, margin)
            equiv_rows.append(out)

        if "dynamic" in weight_cache:
            try:
                w_a = weight_cache["dynamic"]
                w_b = weight_cache["noprob"] if "no-probability" in comp_name and "noprob" in weight_cache else weight_cache["constant"]

                ca = cash_series_from_weights(w_a)
                cb = cash_series_from_weights(w_b)

                ca = ca.loc[(ca.index >= STRICT_START) & (ca.index <= STRICT_END)]
                cb = cb.loc[(cb.index >= STRICT_START) & (cb.index <= STRICT_END)]

                avg_cash_row = bootstrap_mean_diff(
                    ca, cb, "avg_cash",
                    n_rep=BOOTSTRAP_REPLICATIONS,
                    block_length=BOOTSTRAP_BLOCK_LENGTH,
                    seed=BOOTSTRAP_RANDOM_SEED + 100 + spec_idx
                )
                avg_cash_row["comparison"] = comp_name
                avg_cash_row["equivalence_margin"] = EQUIV_MARGINS["avg_cash"]
                avg_cash_row["equivalence_result"] = equivalence_status(avg_cash_row, EQUIV_MARGINS["avg_cash"])
                equiv_rows.append(avg_cash_row)

                ba = (ca >= CASH_CAP - 1e-6).astype(float)
                bb = (cb >= CASH_CAP - 1e-6).astype(float)

                bind_row = bootstrap_mean_diff(
                    ba, bb, "cash_cap_binding",
                    n_rep=BOOTSTRAP_REPLICATIONS,
                    block_length=BOOTSTRAP_BLOCK_LENGTH,
                    seed=BOOTSTRAP_RANDOM_SEED + 200 + spec_idx
                )
                bind_row["comparison"] = comp_name
                bind_row["equivalence_margin"] = EQUIV_MARGINS["cash_cap_binding"]
                bind_row["equivalence_result"] = equivalence_status(bind_row, EQUIV_MARGINS["cash_cap_binding"])
                equiv_rows.append(bind_row)

            except Exception as e:
                warnings_list.append(f"Cash equivalence failed for {comp_name}: {e}")

    s43 = pd.DataFrame(equiv_rows)

    if not s43.empty:
        s43 = s43[
            ["comparison", "metric", "observed_difference", "ci95_lower", "ci95_upper",
             "equivalence_margin", "equivalence_result", "bootstrap_replications", "block_length", "n_days"]
        ]

        summary_rows = []
        for comparison, grp in s43.groupby("comparison"):
            tested = int(len(grp))
            passed = int((grp["equivalence_result"] == "Equivalent within margin").sum())
            summary_rows.append({
                "comparison": comparison,
                "metrics_tested": tested,
                "metrics_equivalent": passed,
                "all_tested_metrics_equivalent": bool(tested == passed),
                "equivalence_results": "; ".join([f"{r.metric}:{r.equivalence_result}" for _, r in grp.iterrows()])
            })
        s43b = pd.DataFrame(summary_rows)
    else:
        s43b = pd.DataFrame()

write_table(s43, "table_S43_constant_lambda_equivalence_tests")
write_rounded_table(s43, "table_S43_constant_lambda_equivalence_tests")
write_table(s43b, "table_S43b_constant_lambda_equivalence_summary")

# ============================================================
# 6. Notebook 21B: pre-evaluation constant-lambda calibration
# ============================================================

print("\n" + "=" * 100)
print("Notebook 21B: Pre-evaluation constant-lambda calibration")
print("=" * 100)

RAW_ETF_FILES = {
    "0050": RAW_YF_DIR / "0050_0050_TW.csv",
    "006208": RAW_YF_DIR / "006208_006208_TW.csv",
    "00692": RAW_YF_DIR / "00692_00692_TW.csv",
    "00881": RAW_YF_DIR / "00881_00881_TW.csv",
}

def load_raw_price_for_etf(symbol):
    p = RAW_ETF_FILES[symbol]
    if not p.exists():
        raise FileNotFoundError(f"Missing raw yfinance file for {symbol}: {p}")

    df = read_table_auto(p)
    desired_adj = {
        "0050": "Adj Close_0050.TW",
        "006208": "Adj Close_006208.TW",
        "00692": "Adj Close_00692.TW",
        "00881": "Adj Close_00881.TW",
    }.get(symbol)

    if desired_adj in df.columns:
        col = desired_adj
    else:
        adj_cols = [c for c in df.columns if "adjclose" in normalize_name(c)]
        close_cols = [c for c in df.columns if "close" in normalize_name(c) and "adj" not in normalize_name(c)]
        if adj_cols:
            col = adj_cols[0]
        elif close_cols:
            col = close_cols[0]
        else:
            raise ValueError(f"No close/adjusted-close column found for {symbol} in raw file {p}. Columns: {list(df.columns)[:20]}")

    s = pd.to_numeric(df[col], errors="coerce").dropna()
    s.name = symbol
    return s.sort_index(), p, col

def load_etf_returns():
    prices = []
    source_rows = []
    for sym in ["0050", "006208", "00692", "00881"]:
        s, p, col = load_raw_price_for_etf(sym)
        prices.append(s)
        source_rows.append({
            "asset": sym,
            "path": str(p),
            "column": col,
            "n": len(s),
            "start": str(s.index.min().date()),
            "end": str(s.index.max().date()),
        })
    price_df = pd.concat(prices, axis=1).sort_index()
    ret_df = price_df.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how="all")
    return price_df, ret_df, pd.DataFrame(source_rows)

def latest_fold_deduplicate(proba_df, split_filter=("validation", "test")):
    df = proba_df.copy()
    if split_filter is not None and "split" in df.columns:
        df = df[df["split"].isin(list(split_filter))].copy()

    df = df.reset_index()
    if "date" not in df.columns:
        df = df.rename(columns={df.columns[0]: "date"})
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df[df["date"].notna()].copy()

    if "fold_id" not in df.columns:
        df["fold_id"] = "WF0"

    df["fold_number"] = df["fold_id"].astype(str).str.extract(r"(\d+)", expand=False).fillna("0").astype(int)
    split_priority = {"train": 0, "validation": 1, "test": 2}
    df["split_priority"] = df["split"].map(split_priority).fillna(0).astype(int) if "split" in df.columns else 0

    df = df.sort_values(["date", "split_priority", "fold_number"])
    df = df.drop_duplicates(subset=["date"], keep="last")
    df = df.set_index("date").sort_index()
    df.index.name = "date"
    return df.drop(columns=["fold_number", "split_priority"], errors="ignore")

def load_probability_features():
    if notebook08_input_index is None or not notebook08_input_index.exists():
        raise FileNotFoundError("NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv not found.")

    idx = pd.read_csv(notebook08_input_index)
    p20_path, p60_path = None, None

    for _, row in idx.iterrows():
        target = str(row.get("target_col", ""))
        parquet_path = Path(str(row.get("probability_path_parquet", "")))
        csv_path = Path(str(row.get("probability_path_csv", "")))
        p = parquet_path if parquet_path.exists() else csv_path
        if "20" in target:
            p20_path = p
        elif "60" in target:
            p60_path = p

    if p20_path is None or p60_path is None:
        raise ValueError("Could not locate both 20d and 60d probability files.")

    p20 = latest_fold_deduplicate(read_table_auto(p20_path), split_filter=("validation", "test"))
    p60 = latest_fold_deduplicate(read_table_auto(p60_path), split_filter=("validation", "test"))

    proba_cols = [f"proba_class_{i}" for i in range(5)]
    for c in proba_cols:
        if c not in p20.columns or c not in p60.columns:
            raise ValueError(f"Missing probability column {c}.")

    common = p20.index.intersection(p60.index).sort_values()
    p20 = p20.loc[common, proba_cols].astype(float)
    p60 = p60.loc[common, proba_cols].astype(float)

    blend = ALPHA_20 * p20.values + ALPHA_60 * p60.values
    blend = np.nan_to_num(blend, nan=0.0)
    rs = blend.sum(axis=1, keepdims=True)
    bad = rs[:, 0] <= 0
    if bad.any():
        blend[bad, :] = 1.0 / 5.0
        rs = blend.sum(axis=1, keepdims=True)
    blend = blend / rs

    eps = 1e-12
    entropy = -np.sum(np.clip(blend, eps, 1.0) * np.log(np.clip(blend, eps, 1.0)), axis=1) / np.log(5)
    bearish = blend[:, 0] + blend[:, 1]
    dynamic_lambda = LAMBDA0 * (1.0 + GAMMA_U * entropy + GAMMA_B * bearish)

    pf = pd.DataFrame({
        "U_t": entropy,
        "B_t": bearish,
        "lambda_dynamic": dynamic_lambda,
    }, index=common)

    for k in range(5):
        pf[f"p{k}"] = blend[:, k]

    return pf, p20_path, p60_path

def make_monthly_rebalance_dates(index):
    index = pd.DatetimeIndex(index).sort_values()
    if len(index) == 0:
        return index
    ser = pd.Series(index=index, data=index)
    if REBALANCE_CONVENTION == "month_start":
        return pd.DatetimeIndex(ser.groupby(index.to_period("M")).min().values).sort_values()
    return pd.DatetimeIndex(ser.groupby(index.to_period("M")).max().values).sort_values()

def cap_and_normalize_weights(w):
    w = np.asarray(w, dtype=float).copy()
    caps = np.array([GENERAL_ETF_CAP, GENERAL_ETF_CAP, GENERAL_ETF_CAP, ETF_00881_CAP, CASH_CAP], dtype=float)

    w = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
    w = np.maximum(w, 0.0)
    w = np.minimum(w, caps)

    for _ in range(30):
        gap = 1.0 - w.sum()
        if abs(gap) < 1e-10:
            break
        if gap > 0:
            capacity = caps - w
            capacity[capacity < 0] = 0
            if capacity.sum() <= 1e-12:
                break
            w += gap * capacity / capacity.sum()
            w = np.minimum(w, caps)
        else:
            positive = w > 0
            if positive.sum() == 0:
                break
            w[positive] += gap * w[positive] / w[positive].sum()
            w = np.maximum(w, 0.0)

    if abs(w.sum() - 1.0) > 1e-6:
        w = np.array([0.10, 0.10, 0.10, 0.10, 0.60])
    return w

def optimize_weight(mu, sigma, lam, prev_w):
    caps = np.array([GENERAL_ETF_CAP, GENERAL_ETF_CAP, GENERAL_ETF_CAP, ETF_00881_CAP, CASH_CAP], dtype=float)
    bounds = [(0.0, caps[i]) for i in range(5)]
    cons = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    x0 = cap_and_normalize_weights(prev_w)

    if not SCIPY_AVAILABLE:
        return x0, {
            "solver_success": False,
            "fallback_used": True,
            "message": "scipy unavailable",
            "objective_value": np.nan,
            "equality_residual": abs(x0.sum() - 1.0),
            "max_bound_violation": 0.0,
        }

    def obj(w):
        return -(
            float(mu @ w)
            - float(lam) * float(w.T @ sigma @ w)
            - float(RHO) * float(np.sum((w - prev_w) ** 2))
        )

    try:
        res = minimize(
            obj,
            x0,
            method="SLSQP",
            bounds=bounds,
            constraints=cons,
            options={"maxiter": 300, "ftol": 1e-10, "disp": False},
        )
        if res.success and np.isfinite(res.fun):
            w = cap_and_normalize_weights(res.x)
            success, fallback, msg, objective = True, False, str(res.message), -float(res.fun)
        else:
            w = x0
            success, fallback, msg, objective = False, True, str(res.message), np.nan
    except Exception as e:
        w = x0
        success, fallback, msg, objective = False, True, repr(e)[:160], np.nan

    eq_resid = abs(float(w.sum() - 1.0))
    lb_viol = float(max(0.0, -np.min(w)))
    ub_viol = float(max(0.0, np.max(w - caps)))
    return w, {
        "solver_success": bool(success),
        "fallback_used": bool(fallback),
        "message": msg,
        "objective_value": objective,
        "equality_residual": eq_resid,
        "max_bound_violation": max(lb_viol, ub_viol),
    }

def estimate_moments(etf_returns, dt):
    past = etf_returns.loc[etf_returns.index < dt, ["0050", "006208", "00692", "00881"]].dropna()
    if len(past) < max(MU_LOOKBACK, COV_LOOKBACK):
        return None, None

    r_mu = past.tail(MU_LOOKBACK)
    r_cov = past.tail(COV_LOOKBACK)

    mean_daily = r_mu.mean().values
    cumulative = (1.0 + r_mu).prod().values
    momentum_daily = np.sign(cumulative - 1.0) * (np.abs(cumulative - 1.0) / MU_LOOKBACK)
    risky_mu = (1.0 - MEAN_SHRINKAGE) * ((1.0 - MOMENTUM_WEIGHT) * mean_daily + MOMENTUM_WEIGHT * momentum_daily)

    cov = r_cov.cov().values
    if not np.all(np.isfinite(cov)):
        return None, None

    eps = max(1e-8, np.nanmean(np.diag(cov)) * 1e-4)
    cov = cov + np.eye(4) * eps

    mu = np.concatenate([risky_mu, [0.0]])
    sigma = np.zeros((5, 5))
    sigma[:4, :4] = cov
    sigma[4, 4] = 1e-12
    return mu, sigma

def run_self_contained_aurora(etf_returns, prob_features, start_date, end_date, mode, constant_lambda=None, annual_cash_return=0.0):
    dates = etf_returns.loc[(etf_returns.index >= start_date) & (etf_returns.index <= end_date)].index
    dates = pd.DatetimeIndex(dates).sort_values()
    if len(dates) == 0:
        raise ValueError(f"No ETF return dates in period {start_date} to {end_date}.")

    pf = prob_features.reindex(dates).ffill().bfill()
    rebalance_dates = set(make_monthly_rebalance_dates(dates))

    current_w = np.array([0.10, 0.10, 0.10, 0.10, 0.60], dtype=float)
    daily_cash_return = (1.0 + annual_cash_return) ** (1.0 / ANNUALIZATION_DAYS) - 1.0

    ret_rows, weight_rows, diag_rows = [], [], []

    for dt in dates:
        tc = 0.0
        if dt in rebalance_dates:
            mu, sigma = estimate_moments(etf_returns, dt)
            if mu is None or sigma is None:
                new_w = current_w.copy()
                diag = {
                    "solver_success": False,
                    "fallback_used": True,
                    "message": "insufficient moment history",
                    "objective_value": np.nan,
                    "equality_residual": abs(new_w.sum() - 1.0),
                    "max_bound_violation": 0.0,
                }
                lam_value = np.nan
            else:
                if mode == "dynamic":
                    lam_value = float(pf.loc[dt, "lambda_dynamic"])
                else:
                    lam_value = float(constant_lambda)
                new_w, diag = optimize_weight(mu, sigma, lam_value, current_w)

            turnover = float(np.sum(np.abs(new_w - current_w)))
            tc = turnover * TRANSACTION_COST_RATE
            current_w = new_w.copy()

            diag_rows.append({
                "date": dt,
                "mode": mode,
                "lambda_value": lam_value,
                "turnover": turnover,
                "transaction_cost": tc,
                "cash_weight": current_w[4],
                "at_cash_cap": float(current_w[4] >= CASH_CAP - 1e-6),
                "at_00881_cap": float(current_w[3] >= ETF_00881_CAP - 1e-6),
                "at_any_etf_cap": float(np.any(current_w[:4] >= np.array([GENERAL_ETF_CAP, GENERAL_ETF_CAP, GENERAL_ETF_CAP, ETF_00881_CAP]) - 1e-6)),
                **diag,
            })

        r_assets = etf_returns.loc[dt, ["0050", "006208", "00692", "00881"]].fillna(0.0).values
        r_day = float(np.dot(current_w[:4], r_assets) + current_w[4] * daily_cash_return - tc)

        ret_rows.append({"date": dt, "return": r_day})
        weight_rows.append({
            "date": dt,
            "0050": current_w[0],
            "006208": current_w[1],
            "00692": current_w[2],
            "00881": current_w[3],
            "cash": current_w[4],
        })

    ret = pd.DataFrame(ret_rows).set_index("date")["return"]
    weights = pd.DataFrame(weight_rows).set_index("date")
    diagnostics = pd.DataFrame(diag_rows).set_index("date") if diag_rows else pd.DataFrame()
    return ret, weights, diagnostics

generated_returns = {}
generated_weights = {}
generated_diags = {}

try:
    price_df, etf_returns, etf_source_df = load_etf_returns()
    write_table(etf_source_df, "notebook21_etf_price_source_inventory")

    prob_features, prob20_path, prob60_path = load_probability_features()
    prob_features.reset_index().rename(columns={"index": "date"}).to_csv(TABLE_RUN_DIR / "notebook21_probability_features_used_for_pre_eval_lambda.csv", index=False)

    common_dates = etf_returns.index.intersection(prob_features.index).sort_values()
    pre_eval_dates = common_dates[common_dates < STRICT_START]
    strict_dates = common_dates[(common_dates >= STRICT_START) & (common_dates <= STRICT_END)]

    if len(pre_eval_dates) < 60:
        raise ValueError(f"Too few pre-evaluation OOS dates for calibration: {len(pre_eval_dates)}")
    if len(strict_dates) < 100:
        raise ValueError(f"Too few strict-test dates for evaluation: {len(strict_dates)}")

    cal_start, cal_end = pre_eval_dates.min(), pre_eval_dates.max()
    test_start, test_end = strict_dates.min(), strict_dates.max()

    print("Pre-evaluation calibration window:", cal_start.date(), "to", cal_end.date(), "n=", len(pre_eval_dates))
    print("Strict-test evaluation window:", test_start.date(), "to", test_end.date(), "n=", len(strict_dates))

    dyn_cal_ret, dyn_cal_w, dyn_cal_diag = run_self_contained_aurora(
        etf_returns, prob_features, cal_start, cal_end, mode="dynamic", annual_cash_return=0.0
    )
    target_cal_avg_cash = float(dyn_cal_w["cash"].mean())

    grid_rows = []
    best = None

    for lam in LAMBDA_GRID:
        ret_l, w_l, diag_l = run_self_contained_aurora(
            etf_returns, prob_features, cal_start, cal_end, mode="constant", constant_lambda=lam, annual_cash_return=0.0
        )
        avg_cash = float(w_l["cash"].mean())
        err = abs(avg_cash - target_cal_avg_cash)
        perf = performance_metrics_from_returns(ret_l)

        row = {
            "lambda": float(lam),
            "calibration_avg_cash": avg_cash,
            "dynamic_calibration_avg_cash": target_cal_avg_cash,
            "cash_match_abs_error": err,
            "calibration_total_return": perf["total_return"],
            "calibration_sharpe": perf["sharpe"],
            "calibration_sortino": perf["sortino"],
            "calibration_max_drawdown": perf["max_drawdown"],
        }
        grid_rows.append(row)
        if best is None or err < best["cash_match_abs_error"]:
            best = row

    selected_lambda_pre_eval = float(best["lambda"])
    grid_df = pd.DataFrame(grid_rows).sort_values(["cash_match_abs_error", "lambda"])
    write_table(grid_df, "table_S44_pre_evaluation_constant_lambda_grid_search")
    write_rounded_table(grid_df, "table_S44_pre_evaluation_constant_lambda_grid_search")

    dyn_test_ret, dyn_test_w, dyn_test_diag = run_self_contained_aurora(
        etf_returns, prob_features, test_start, test_end, mode="dynamic", annual_cash_return=0.0
    )
    pre_const_test_ret, pre_const_test_w, pre_const_test_diag = run_self_contained_aurora(
        etf_returns, prob_features, test_start, test_end, mode="constant", constant_lambda=selected_lambda_pre_eval, annual_cash_return=0.0
    )
    pre_noprob_test_ret, pre_noprob_test_w, pre_noprob_test_diag = run_self_contained_aurora(
        etf_returns, prob_features, test_start, test_end, mode="constant_noprob", constant_lambda=selected_lambda_pre_eval, annual_cash_return=0.0
    )

    generated_returns["Notebook21 dynamic AURORA"] = dyn_test_ret
    generated_returns[f"Notebook21 pre-eval constant-lambda {selected_lambda_pre_eval:g}"] = pre_const_test_ret
    generated_returns[f"Notebook21 pre-eval no-probability constant-lambda {selected_lambda_pre_eval:g}"] = pre_noprob_test_ret

    generated_weights["Notebook21 dynamic AURORA"] = dyn_test_w
    generated_weights[f"Notebook21 pre-eval constant-lambda {selected_lambda_pre_eval:g}"] = pre_const_test_w
    generated_weights[f"Notebook21 pre-eval no-probability constant-lambda {selected_lambda_pre_eval:g}"] = pre_noprob_test_w

    generated_diags["Notebook21 dynamic AURORA"] = dyn_test_diag
    generated_diags[f"Notebook21 pre-eval constant-lambda {selected_lambda_pre_eval:g}"] = pre_const_test_diag
    generated_diags[f"Notebook21 pre-eval no-probability constant-lambda {selected_lambda_pre_eval:g}"] = pre_noprob_test_diag

    gen_ret_df = pd.DataFrame(generated_returns)
    save_frame(gen_ret_df, RETURNS_DIR / "notebook21_pre_eval_constant_lambda_returns.parquet")
    gen_ret_df.to_csv(RETURNS_DIR / "notebook21_pre_eval_constant_lambda_returns.csv")

    gen_w_long = []
    for name, wdf in generated_weights.items():
        temp = wdf.copy()
        temp["strategy"] = name
        temp = temp.reset_index()
        gen_w_long.append(temp)
    gen_w_long = pd.concat(gen_w_long, ignore_index=True)
    gen_w_long.to_csv(WEIGHTS_DIR / "notebook21_pre_eval_constant_lambda_weights.csv", index=False)

    perf_rows = []
    for name, ret in generated_returns.items():
        w = generated_weights[name]
        perf = performance_metrics_from_returns(ret)
        perf_rows.append({
            "strategy_control": name,
            "calibration_source": "pre-evaluation out-of-sample prediction dates",
            "selected_lambda": selected_lambda_pre_eval if "constant-lambda" in name else np.nan,
            "test_total_return": perf["total_return"],
            "test_sharpe": perf["sharpe"],
            "test_sortino": perf["sortino"],
            "test_max_drawdown": perf["max_drawdown"],
            "avg_cash": float(w["cash"].mean()),
            "days_at_cash_cap": float((w["cash"] >= CASH_CAP - 1e-6).mean()),
            "n_days": int(len(ret)),
        })

    s44 = pd.DataFrame([{
        "status": "completed",
        "calibration_window_start": str(cal_start.date()),
        "calibration_window_end": str(cal_end.date()),
        "calibration_n_days": int(len(pre_eval_dates)),
        "strict_test_window_start": str(test_start.date()),
        "strict_test_window_end": str(test_end.date()),
        "strict_test_n_days": int(len(strict_dates)),
        "dynamic_calibration_avg_cash": target_cal_avg_cash,
        "selected_pre_evaluation_lambda": selected_lambda_pre_eval,
        "cash_match_abs_error": float(best["cash_match_abs_error"]),
        "lambda_grid_min": float(np.min(LAMBDA_GRID)),
        "lambda_grid_max": float(np.max(LAMBDA_GRID)),
        "lambda_grid_step": float(np.diff(LAMBDA_GRID).min()) if len(LAMBDA_GRID) > 1 else np.nan,
        "implementation_note": "Self-contained AURORA-compatible optimizer reconstruction using raw ETF adjusted-close returns and pre-evaluation OOS probabilities.",
    }])

    s44b = pd.DataFrame(perf_rows)

except Exception as e:
    warnings_list.append(f"Pre-evaluation constant-lambda calibration failed: {repr(e)}")
    selected_lambda_pre_eval = np.nan
    s44 = pd.DataFrame([{
        "status": "failed",
        "reason": repr(e),
        "required_inputs": "raw ETF adjusted-close files in data/raw_yfinance, Notebook08 probability input index, selected 20d/60d probability files",
    }])
    s44b = pd.DataFrame()

write_table(s44, "table_S44_pre_evaluation_constant_lambda_calibration")
write_rounded_table(s44, "table_S44_pre_evaluation_constant_lambda_calibration")
write_table(s44b, "table_S44b_pre_evaluation_lambda_performance")
if not s44b.empty:
    write_rounded_table(s44b, "table_S44b_pre_evaluation_lambda_performance")

# ============================================================
# 7. Notebook 21C: optimizer and constraint diagnostics
# ============================================================

print("\n" + "=" * 100)
print("Notebook 21C: Optimizer and constraint diagnostics")
print("=" * 100)

def summarize_diag_frame(diag_df, source_label):
    if diag_df is None or len(diag_df) == 0:
        return None

    df = diag_df.copy()
    n = len(df)

    def find_col_any(tokens_any):
        for c in df.columns:
            nc = normalize_name(c)
            if any(tok in nc for tok in tokens_any):
                return c
        return None

    def find_col_all(tokens_all):
        for c in df.columns:
            nc = normalize_name(c)
            if all(tok in nc for tok in tokens_all):
                return c
        return None

    success_col = find_col_any(["success", "solverstatus"])
    fallback_col = find_col_any(["fallback"])
    eq_col = find_col_all(["equality", "residual"])
    bound_col = find_col_all(["bound", "violation"])
    cash_col = find_col_all(["cash", "weight"])
    turnover_col = find_col_any(["turnover"])

    success_rate = np.nan
    if success_col:
        sv = df[success_col]
        if sv.dtype == bool:
            success_rate = float(sv.mean())
        else:
            success_rate = float(pd.to_numeric(sv, errors="coerce").fillna(0).mean())

    fallback_count = np.nan
    if fallback_col:
        fv = df[fallback_col]
        if fv.dtype == bool:
            fallback_count = int(fv.sum())
        else:
            fallback_count = int(pd.to_numeric(fv, errors="coerce").fillna(0).sum())

    mean_eq = np.nan
    max_eq = np.nan
    if eq_col:
        vals = pd.to_numeric(df[eq_col], errors="coerce").abs()
        mean_eq = float(vals.mean())
        max_eq = float(vals.max())

    max_bound = np.nan
    if bound_col:
        max_bound = float(pd.to_numeric(df[bound_col], errors="coerce").abs().max())

    avg_turnover = np.nan
    if turnover_col:
        avg_turnover = float(pd.to_numeric(df[turnover_col], errors="coerce").mean())

    cash_cap_frac = np.nan
    if cash_col:
        cash = pd.to_numeric(df[cash_col], errors="coerce")
        cash_cap_frac = float((cash >= CASH_CAP - 1e-6).mean())

    return {
        "diagnostic_source": source_label,
        "rebalance_dates_evaluated": int(n),
        "successful_optimizer_solves": int(round(success_rate * n)) if np.isfinite(success_rate) else np.nan,
        "solver_success_rate": success_rate,
        "fallback_allocations": fallback_count,
        "fallback_rate": float(fallback_count / n) if np.isfinite(fallback_count) and n > 0 else np.nan,
        "mean_equality_residual_abs": mean_eq,
        "max_equality_residual_abs": max_eq,
        "maximum_bound_violation": max_bound,
        "fraction_days_at_cash_cap": cash_cap_frac,
        "average_turnover": avg_turnover,
    }

diag_summary_rows = []

if n18_diag_path is not None:
    try:
        raw_diag = read_table_auto(n18_diag_path)
        source_col = next((c for c in ["strategy_control", "strategy", "strategy_name", "control"] if c in raw_diag.columns), None)
        if source_col:
            for name, sub in raw_diag.groupby(source_col):
                ss = summarize_diag_frame(sub, f"Notebook18 diagnostics: {name}")
                if ss:
                    diag_summary_rows.append(ss)
        else:
            ss = summarize_diag_frame(raw_diag, "Notebook18 all_notebook18_diagnostics")
            if ss:
                diag_summary_rows.append(ss)
        raw_diag.head(200).to_csv(DIAG_DIR / "notebook18_diagnostics_preview.csv")
        pd.DataFrame({"columns": list(raw_diag.columns)}).to_csv(DIAG_DIR / "notebook18_diagnostics_columns.csv", index=False)
    except Exception as e:
        warnings_list.append(f"Could not summarize Notebook18 diagnostics: {e}")

for name, diag in generated_diags.items():
    ss = summarize_diag_frame(diag, name)
    if ss:
        diag_summary_rows.append(ss)

s45 = pd.DataFrame(diag_summary_rows)
if s45.empty:
    s45 = pd.DataFrame([{"status": "No optimizer diagnostic rows available."}])

write_table(s45, "table_S45_optimizer_reliability_diagnostics")
write_rounded_table(s45, "table_S45_optimizer_reliability_diagnostics")

binding_rows = []

def add_binding_row(name, wdf, source):
    try:
        wdf = standardize_weight_columns(wdf)
        if len(wdf) == 0:
            return
        row = {
            "strategy_control": name,
            "source": source,
            "n_days": int(len(wdf)),
            "avg_cash": float(wdf["cash"].mean()) if "cash" in wdf.columns else np.nan,
            "median_cash": float(wdf["cash"].median()) if "cash" in wdf.columns else np.nan,
            "fraction_days_at_cash_cap": float((wdf["cash"] >= CASH_CAP - 1e-6).mean()) if "cash" in wdf.columns else np.nan,
            "fraction_days_at_0050_cap": float((wdf["0050"] >= GENERAL_ETF_CAP - 1e-6).mean()) if "0050" in wdf.columns else np.nan,
            "fraction_days_at_006208_cap": float((wdf["006208"] >= GENERAL_ETF_CAP - 1e-6).mean()) if "006208" in wdf.columns else np.nan,
            "fraction_days_at_00692_cap": float((wdf["00692"] >= GENERAL_ETF_CAP - 1e-6).mean()) if "00692" in wdf.columns else np.nan,
            "fraction_days_at_00881_cap": float((wdf["00881"] >= ETF_00881_CAP - 1e-6).mean()) if "00881" in wdf.columns else np.nan,
        }
        risky = [c for c in ["0050", "006208", "00692", "00881"] if c in wdf.columns]
        if risky:
            caps = {"0050": GENERAL_ETF_CAP, "006208": GENERAL_ETF_CAP, "00692": GENERAL_ETF_CAP, "00881": ETF_00881_CAP}
            at_any = np.zeros(len(wdf), dtype=bool)
            for c in risky:
                at_any = at_any | (wdf[c].values >= caps[c] - 1e-6)
            row["fraction_days_at_any_etf_cap"] = float(at_any.mean())
            row["avg_equity_exposure"] = float(wdf[risky].sum(axis=1).mean())
        binding_rows.append(row)
    except Exception as e:
        warnings_list.append(f"Constraint-binding diagnostics failed for {name}: {e}")

if n18_weights_path is not None:
    try:
        wdf_all = read_table_auto(n18_weights_path)
        for name, labels in [
            ("Original dynamic AURORA", DYN_LABELS),
            ("Ex post constant-lambda AURORA", CONST_LABELS),
            ("Ex post no-probability constant-lambda AURORA", NOPROB_LABELS),
        ]:
            try:
                w = extract_strategy_weights(wdf_all, labels)
                w = w.loc[(w.index >= STRICT_START) & (w.index <= STRICT_END)]
                add_binding_row(name, w, "Notebook18 weights")
            except Exception as e:
                warnings_list.append(f"Could not extract Notebook18 weights for {name}: {e}")
    except Exception as e:
        warnings_list.append(f"Could not read Notebook18 weights: {e}")

for name, wdf in generated_weights.items():
    add_binding_row(name, wdf, "Notebook21 self-contained weights")

s45b = pd.DataFrame(binding_rows)
if s45b.empty:
    s45b = pd.DataFrame([{"status": "No weight rows available for constraint-binding diagnostics."}])

write_table(s45b, "table_S45b_constraint_binding_diagnostics")
write_rounded_table(s45b, "table_S45b_constraint_binding_diagnostics")

# ============================================================
# 8. Notebook 22: cash-return / nonzero-RF sensitivity
# ============================================================

print("\n" + "=" * 100)
print("Notebook 22: Cash-return and risk-free-rate sensitivity")
print("=" * 100)

strategy_cash_inputs = {}

if n18_returns_path is not None and n18_weights_path is not None:
    try:
        n18_returns_df = read_table_auto(n18_returns_path)
        n18_weights_df = read_table_auto(n18_weights_path)

        for name, labels in [
            ("Original dynamic AURORA", DYN_LABELS),
            ("Ex post constant-lambda AURORA", CONST_LABELS),
            ("Ex post no-probability constant-lambda AURORA", NOPROB_LABELS),
        ]:
            try:
                r = extract_strategy_series(n18_returns_df, labels)
                w = extract_strategy_weights(n18_weights_df, labels)
                cash = cash_series_from_weights(w)
                common = r.index.intersection(cash.index).sort_values()
                common = common[(common >= STRICT_START) & (common <= STRICT_END)]
                if len(common) == 319:
                    strategy_cash_inputs[name] = {
                        "base_return": r.loc[common],
                        "cash_weight": cash.loc[common],
                        "source": "Notebook18 returns and weights",
                    }
                else:
                    warnings_list.append(f"Skipped {name} from S46 because n_days={len(common)}, expected 319.")
            except Exception as e:
                warnings_list.append(f"Could not prepare S46 input for {name}: {e}")

    except Exception as e:
        warnings_list.append(f"Notebook18 cash-sensitivity preparation failed: {e}")

# Optional source-aware cash sensitivity is deliberately disabled by default.
if INCLUDE_SOURCE_AWARE_CASH_SENSITIVITY:
    warnings_list.append(
        "INCLUDE_SOURCE_AWARE_CASH_SENSITIVITY=True, but this notebook does not automatically include source-aware rows unless exact source-aware cash weights are manually integrated."
    )
else:
    warnings_list.append(
        "Source-aware cash-return sensitivity skipped by design because exact source-aware cash-weight files were not used. "
        "Only valid Notebook18 dynamic/constant-lambda diagnostic strategies are reported."
    )

cash_sens_rows = []

for strategy_name, obj in strategy_cash_inputs.items():
    base = pd.Series(obj["base_return"]).dropna().astype(float)
    cash_w = pd.Series(obj["cash_weight"]).dropna().astype(float)
    common = base.index.intersection(cash_w.index).sort_values()
    common = common[(common >= STRICT_START) & (common <= STRICT_END)]
    base = base.loc[common]
    cash_w = cash_w.loc[common]

    for scenario in CASH_RATE_SCENARIOS:
        annual_cash = float(scenario["annual_cash_return"])
        annual_rf = float(scenario["annual_risk_free_rate"])
        daily_cash = (1.0 + annual_cash) ** (1.0 / ANNUALIZATION_DAYS) - 1.0
        daily_rf = (1.0 + annual_rf) ** (1.0 / ANNUALIZATION_DAYS) - 1.0

        adjusted = base + cash_w * daily_cash
        p0 = performance_metrics_from_returns(adjusted, rf_daily=0.0)
        prf = performance_metrics_from_returns(adjusted, rf_daily=daily_rf)

        cash_sens_rows.append({
            "strategy_control": strategy_name,
            "input_source": obj["source"],
            "cash_return_scenario": scenario["scenario"],
            "annual_cash_return": annual_cash,
            "annual_risk_free_rate": annual_rf,
            "n_days": int(len(adjusted)),
            "avg_cash_weight": float(cash_w.mean()),
            "total_return": p0["total_return"],
            "annual_return": p0["annual_return"],
            "max_drawdown": p0["max_drawdown"],
            "sharpe_zero_rf": p0["sharpe"],
            "sharpe_nonzero_rf": prf["sharpe"],
            "sortino_zero_rf": p0["sortino"],
            "sortino_nonzero_rf": prf["sortino"],
            "calmar": p0["calmar"],
        })

s46 = pd.DataFrame(cash_sens_rows)

# Final hard safety filter to prevent invalid source-aware rows.
if not s46.empty:
    valid_names = [
        "Original dynamic AURORA",
        "Ex post constant-lambda AURORA",
        "Ex post no-probability constant-lambda AURORA",
    ]
    s46 = s46[
        (s46["strategy_control"].isin(valid_names)) &
        (s46["n_days"] == 319) &
        (s46["input_source"].str.contains("Notebook18 returns and weights", na=False))
    ].copy()

if s46.empty:
    s46 = pd.DataFrame([{
        "status": "No valid n=319 Notebook18 return/cash-weight pairs found for cash-return sensitivity.",
    }])

write_table(s46, "table_S46_cash_return_risk_free_sensitivity")
write_rounded_table(s46, "table_S46_cash_return_risk_free_sensitivity")

summary_rows = []

if "strategy_control" in s46.columns and "total_return" in s46.columns:
    for name, grp in s46.groupby("strategy_control"):
        grp = grp.dropna(subset=["total_return"])
        if grp.empty:
            continue
        base_row = grp.loc[grp["cash_return_scenario"] == "zero_cash_return"]
        high_row = grp.loc[grp["annual_cash_return"].idxmax()]

        base_total = float(base_row["total_return"].iloc[0]) if len(base_row) else np.nan
        high_total = float(high_row["total_return"]) if isinstance(high_row, pd.Series) else np.nan

        summary_rows.append({
            "strategy_control": name,
            "avg_cash_weight": float(grp["avg_cash_weight"].mean()),
            "base_total_return_zero_cash": base_total,
            "highest_cash_scenario_total_return": high_total,
            "total_return_change_highest_minus_base": high_total - base_total if np.isfinite(base_total) and np.isfinite(high_total) else np.nan,
            "base_sharpe_zero_rf": float(base_row["sharpe_zero_rf"].iloc[0]) if len(base_row) else np.nan,
            "highest_cash_scenario_sharpe_nonzero_rf": float(high_row["sharpe_nonzero_rf"]) if isinstance(high_row, pd.Series) else np.nan,
            "interpretation": "Cash yield raises total return for high-cash variants; using the same nonzero risk-free rate lowers excess-return Sharpe.",
        })

s46b = pd.DataFrame(summary_rows)
if s46b.empty:
    s46b = pd.DataFrame([{"status": "Cash-return sensitivity summary not generated."}])

write_table(s46b, "table_S46b_cash_return_sensitivity_summary")
write_rounded_table(s46b, "table_S46b_cash_return_sensitivity_summary")

# ============================================================
# 9. Interpretation helper
# ============================================================

interpret_rows = []

if not s43.empty:
    for _, row in s43.iterrows():
        interpret_rows.append({
            "section": "S43 equivalence",
            "item": f"{row['comparison']} | {row['metric']}",
            "finding": row["equivalence_result"],
            "evidence": f"Observed diff={row['observed_difference']:.6f}; CI=[{row['ci95_lower']:.6f},{row['ci95_upper']:.6f}]; margin=±{row['equivalence_margin']:.6f}.",
        })

if not s44b.empty and "strategy_control" in s44b.columns:
    for _, row in s44b.iterrows():
        interpret_rows.append({
            "section": "S44 pre-evaluation lambda",
            "item": row["strategy_control"],
            "finding": f"Strict-test performance under selected pre-evaluation lambda {row['selected_lambda']}",
            "evidence": f"TR={row['test_total_return']:.4f}; Sharpe={row['test_sharpe']:.4f}; MDD={row['test_max_drawdown']:.4f}; avg cash={row['avg_cash']:.4f}.",
        })

if not s46b.empty and "strategy_control" in s46b.columns:
    for _, row in s46b.iterrows():
        if "total_return_change_highest_minus_base" in row and pd.notna(row["total_return_change_highest_minus_base"]):
            interpret_rows.append({
                "section": "S46 cash-return sensitivity",
                "item": row["strategy_control"],
                "finding": "Nonzero cash return changes absolute performance but not the dynamic-vs-constant attribution conclusion.",
                "evidence": f"Avg cash={row['avg_cash_weight']:.4f}; highest-minus-base TR change={row['total_return_change_highest_minus_base']:.4f}.",
            })

interpret_df = pd.DataFrame(interpret_rows)
write_table(interpret_df, "notebook21_22_interpretation_helper")

# ============================================================
# 10. Validation reports and manifest
# ============================================================

validation_21 = {
    "project_code": PROJECT_CODE,
    "notebook": "21_revised_equivalence_pre_eval_lambda_optimizer_diagnostics",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "inputs_found": {
        "notebook18_returns": str(n18_returns_path) if n18_returns_path else None,
        "notebook18_weights": str(n18_weights_path) if n18_weights_path else None,
        "notebook18_diagnostics": str(n18_diag_path) if n18_diag_path else None,
        "notebook08_input_index": str(notebook08_input_index) if notebook08_input_index else None,
        "raw_yfinance_dir": str(RAW_YF_DIR),
    },
    "equivalence_settings": {
        "bootstrap_replications": BOOTSTRAP_REPLICATIONS,
        "block_length": BOOTSTRAP_BLOCK_LENGTH,
        "equivalence_margins": EQUIV_MARGINS,
    },
    "pre_evaluation_lambda_settings": {
        "lambda_grid_min": float(np.min(LAMBDA_GRID)),
        "lambda_grid_max": float(np.max(LAMBDA_GRID)),
        "lambda_grid_step": float(np.diff(LAMBDA_GRID).min()) if len(LAMBDA_GRID) > 1 else None,
        "selected_lambda_pre_eval": float(selected_lambda_pre_eval) if np.isfinite(selected_lambda_pre_eval) else None,
        "rebalance_convention": REBALANCE_CONVENTION,
        "transaction_cost_bps": TRANSACTION_COST_BPS,
    },
    "outputs": {
        "table_S43": str(TABLE_RUN_DIR / "table_S43_constant_lambda_equivalence_tests.csv"),
        "table_S43b": str(TABLE_RUN_DIR / "table_S43b_constant_lambda_equivalence_summary.csv"),
        "table_S44": str(TABLE_RUN_DIR / "table_S44_pre_evaluation_constant_lambda_calibration.csv"),
        "table_S44b": str(TABLE_RUN_DIR / "table_S44b_pre_evaluation_lambda_performance.csv"),
        "table_S45": str(TABLE_RUN_DIR / "table_S45_optimizer_reliability_diagnostics.csv"),
        "table_S45b": str(TABLE_RUN_DIR / "table_S45b_constraint_binding_diagnostics.csv"),
    },
    "warnings": warnings_list,
}

validation_22 = {
    "project_code": PROJECT_CODE,
    "notebook": "22_revised_cash_return_risk_free_sensitivity",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "cash_rate_scenarios": CASH_RATE_SCENARIOS,
    "include_source_aware_cash_sensitivity": INCLUDE_SOURCE_AWARE_CASH_SENSITIVITY,
    "valid_strategy_rows_only": [
        "Original dynamic AURORA",
        "Ex post constant-lambda AURORA",
        "Ex post no-probability constant-lambda AURORA",
    ],
    "outputs": {
        "table_S46": str(TABLE_RUN_DIR / "table_S46_cash_return_risk_free_sensitivity.csv"),
        "table_S46b": str(TABLE_RUN_DIR / "table_S46b_cash_return_sensitivity_summary.csv"),
    },
    "warnings": warnings_list,
}

save_json(REPORT_RUN_DIR / "NOTEBOOK21_validation_report.json", validation_21)
save_json(REPORT_RUN_DIR / "NOTEBOOK22_validation_report.json", validation_22)
save_json(REPORT_DIR / f"NOTEBOOK21_validation_report_{RUN_ID}.json", validation_21)
save_json(REPORT_DIR / f"NOTEBOOK22_validation_report_{RUN_ID}.json", validation_22)

manifest_df = make_file_manifest(RUN_ROOT)
manifest_df.to_csv(REPORT_RUN_DIR / "NOTEBOOK21_22_file_manifest_SHA256.csv", index=False)
manifest_df.to_csv(REPORT_DIR / f"NOTEBOOK21_22_file_manifest_SHA256_{RUN_ID}.csv", index=False)

# ============================================================
# 11. Final summary
# ============================================================

print("\n" + "=" * 100)
print("REVISED NOTEBOOKS 21 + 22 COMPLETE")
print("=" * 100)
print("RUN_ID:", RUN_ID)
print("RUN_ROOT:", RUN_ROOT)

print("\nKey outputs:")
for name in [
    "table_S43_constant_lambda_equivalence_tests.csv",
    "table_S43b_constant_lambda_equivalence_summary.csv",
    "table_S44_pre_evaluation_constant_lambda_calibration.csv",
    "table_S44b_pre_evaluation_lambda_performance.csv",
    "table_S45_optimizer_reliability_diagnostics.csv",
    "table_S45b_constraint_binding_diagnostics.csv",
    "table_S46_cash_return_risk_free_sensitivity.csv",
    "table_S46b_cash_return_sensitivity_summary.csv",
    "notebook21_22_interpretation_helper.csv",
]:
    print(TABLE_RUN_DIR / name)

print("\nWarnings:")
if warnings_list:
    for w in warnings_list:
        print("-", w)
else:
    print("None")

print("\nPreview S43:")
print(s43.round(6).to_string(index=False) if not s43.empty else "No S43 rows.")

print("\nPreview S43b:")
print(s43b.to_string(index=False) if not s43b.empty else "No S43b rows.")

print("\nPreview S44:")
print(s44.round(6).to_string(index=False) if not s44.empty else "No S44 rows.")

print("\nPreview S44b:")
print(s44b.round(6).to_string(index=False) if not s44b.empty else "No S44b rows.")

print("\nPreview S45:")
print(s45.round(6).to_string(index=False) if not s45.empty else "No S45 rows.")

print("\nPreview S45b:")
print(s45b.round(6).to_string(index=False) if not s45b.empty else "No S45b rows.")

print("\nPreview S46:")
print(s46.round(6).to_string(index=False) if not s46.empty else "No S46 rows.")

print("\nPreview S46b:")
print(s46b.round(6).to_string(index=False) if not s46b.empty else "No S46b rows.")

print("\nInterpretation helper:")
print(interpret_df.to_string(index=False) if not interpret_df.empty else "No interpretation rows.")
print("=" * 100)

Mounted at /content/drive
AURORA-TWETF Revised Notebooks 21 + 22
RUN_ID: 20260722_133503
RUN_ROOT: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/equivalence_lambda_cash_sensitivity/run_20260722_133503

Locating prior output files
Notebook18 returns: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/aurora_exposure_matched_lambda_reduced_universe/run_20260717_082548/returns/all_notebook18_returns.csv
Notebook18 weights: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/aurora_exposure_matched_lambda_reduced_universe/run_20260717_082548/weights/all_notebook18_weights.parquet
Notebook18 diagnostics: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/aurora_exposure_matched_lambda_reduced_universe/run_20260717_082548/diagnostics/all_notebook18_diagnostics.csv
Source-aware return matrix: /content/drive/MyDrive/AURORA_TWETF/outputs/ROMA_AURORA_TWETF/source_aware_unified_paper_comparison/run_20260625_065916/returns/notebook13B_source_aware_strict_test_return_mat

In [1]:
# ============================================================
# AURORA-TWETF Colab environment/package version snapshot
# Single-cell version
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import sys
import platform
import subprocess
import textwrap

# Change this path if your project folder is different
PROJECT_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

# Mount Google Drive if running in Colab
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception as e:
    IN_COLAB = False
    print("Google Drive mount skipped or unavailable:", repr(e))

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

# Packages most relevant to AURORA-TWETF
project_packages = [
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "matplotlib",
    "seaborn",
    "yfinance",
    "pyarrow",
    "openpyxl",
    "joblib",
    "tqdm",
    "xgboost",
    "lightgbm",
]

# Get versions using importlib.metadata
try:
    from importlib.metadata import version, PackageNotFoundError
except Exception:
    from importlib_metadata import version, PackageNotFoundError

timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

summary_lines = []
summary_lines.append("# AURORA-TWETF Environment Snapshot")
summary_lines.append("")
summary_lines.append(f"Generated UTC: {timestamp}")
summary_lines.append(f"Running in Colab: {IN_COLAB}")
summary_lines.append("")
summary_lines.append("## System")
summary_lines.append("")
summary_lines.append(f"Python: {sys.version.replace(chr(10), ' ')}")
summary_lines.append(f"Platform: {platform.platform()}")
summary_lines.append(f"Machine: {platform.machine()}")
summary_lines.append(f"Processor: {platform.processor()}")
summary_lines.append("")
summary_lines.append("## Project package versions")
summary_lines.append("")

requirements_lines = []

for pkg in project_packages:
    try:
        v = version(pkg)
        line = f"{pkg}=={v}"
    except PackageNotFoundError:
        line = f"# {pkg} not installed"
    requirements_lines.append(line)
    summary_lines.append(line)

summary_text = "\n".join(summary_lines)
requirements_text = "\n".join(requirements_lines) + "\n"

# Save clean project package versions
requirements_project_path = PROJECT_ROOT / "requirements_project_versions.txt"
requirements_project_path.write_text(requirements_text, encoding="utf-8")

# Save readable environment snapshot
environment_snapshot_path = PROJECT_ROOT / "ENVIRONMENT_SNAPSHOT.md"
environment_snapshot_path.write_text(summary_text + "\n", encoding="utf-8")

# Save full Colab/runtime package snapshot
requirements_colab_path = PROJECT_ROOT / "requirements_colab_full.txt"
try:
    freeze_result = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        capture_output=True,
        text=True,
        check=True,
    )
    requirements_colab_path.write_text(freeze_result.stdout, encoding="utf-8")
    freeze_status = "saved"
except Exception as e:
    requirements_colab_path.write_text(f"pip freeze failed: {repr(e)}\n", encoding="utf-8")
    freeze_status = f"failed: {repr(e)}"

print("=" * 80)
print("AURORA-TWETF environment snapshot complete")
print("=" * 80)
print(summary_text)
print("\nSaved files:")
print("-", requirements_project_path)
print("-", environment_snapshot_path)
print("-", requirements_colab_path, f"({freeze_status})")
print("=" * 80)

Mounted at /content/drive
AURORA-TWETF environment snapshot complete
# AURORA-TWETF Environment Snapshot

Generated UTC: 2026-07-24T00:18:41Z
Running in Colab: True

## System

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Machine: x86_64
Processor: x86_64

## Project package versions

numpy==2.0.2
pandas==2.2.2
scipy==1.16.3
scikit-learn==1.6.1
matplotlib==3.10.0
seaborn==0.13.2
yfinance==0.2.66
pyarrow==18.1.0
openpyxl==3.1.5
joblib==1.5.3
tqdm==4.67.3
xgboost==3.3.0
lightgbm==4.6.0

Saved files:
- /content/drive/MyDrive/AURORA_TWETF/requirements_project_versions.txt
- /content/drive/MyDrive/AURORA_TWETF/ENVIRONMENT_SNAPSHOT.md
- /content/drive/MyDrive/AURORA_TWETF/requirements_colab_full.txt (saved)
